<a href="https://colab.research.google.com/github/khariharan897/BronzeBadge_GENAI_Assessment/blob/main/Assignment5_Policy_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment 5:
This is the case; you must develop using LLM and RAG. After submission you people must present to some of the panel members.

 Use case: “Policy & Claims Copilot” (Customer support + Claims pre-check)

Goal - Help customers, agents, and claims teams get instant, consistent answers about:

what’s covered / not covered

limits & sub-limits

waiting periods

claim submission steps + timelines

documents needed
…and also do a pre-check of a claim scenario before submission.

This reduces call center load, speeds claim filing, and improves first-time-right submissions.



Why RAG is needed (vs plain LLM)

A plain LLM might “guess” policy terms. With RAG, the assistant:

retrieves the exact relevant clauses from the policy PDF

answers using only those clauses

quotes/links the source section/page (grounded response)

Assignment 5: RAG Pipeline - Sample Code Structure

To build a RAG pipeline, we'll need to install a few libraries. This example uses langchain for orchestration, pypdf for PDF loading, sentence-transformers for embeddings, and faiss-cpu for a local vector store. You might also need a library for your specific LLM integration (e.g., openai, ollama, google-generativeai).

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving PolicyDocument.pdf to PolicyDocument.pdf


In [ ]:
# Install necessary libraries
!pip install -q langchain langchain-community langchain-core langchain-text-splitters pypdf sentence-transformers faiss-cpu
# If you plan to use OpenAI models, uncomment the following:
#!pip install -q openai
# If you plan to use Google Gemini models, uncomment the following:
#!pip install -q google-generativeai

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
import os

# --- Create a dummy PDF file for demonstration ---
# In a real scenario, you would have your actual policy PDFs.
# For this example, we'll create a simple text file and convert it to PDF programmatically.
# (Note: This conversion functionality is not part of standard Python libraries and
# typically requires external tools or libraries like 'reportlab' or 'fpdf'.
# For simplicity, we'll assume a 'dummy_policy.pdf' exists or you'll place one there.)

dummy_pdf_content = """
Insurance Policy Document

Section 1: Coverage
1.1 Medical Expenses: This policy covers medical expenses up to $10,000 per incident.
1.2 Dental Coverage: Dental treatment is covered for accidental injury only, up to $1,000.

Section 2: Limits
2.1 Overall Limit: The total claim limit for this policy is $50,000 per year.
2.2 Sub-limits: Specific sub-limits apply to certain categories as detailed in Appendix A.

Section 3: Waiting Periods
3.1 New Policies: A 30-day waiting period applies for all non-accidental medical claims from the policy start date.
3.2 Pre-existing Conditions: A 90-day waiting period applies for claims related to pre-existing conditions.

Section 4: Claim Submission
4.1 Steps: Claims must be submitted via the online portal within 15 days of the incident.
4.2 Documents: Required documents include: claim form, medical reports, original bills, and proof of payment.

This is a dummy document for demonstration purposes.
"""

# You would typically place your PDF file in a specific directory.
# For this example, let's pretend 'dummy_policy.pdf' is available.
# If you want to make a real PDF, you'd need a library like 'reportlab' or manually create one.
# Example for creating a dummy PDF (requires reportlab, install with `pip install reportlab`)
# from reportlab.pdfgen import canvas
# def create_dummy_pdf(filename, content):
#     c = canvas.Canvas(filename)
#     textobject = c.beginText()
#     textobject.setTextOrigin(10, 800)
#     for line in content.split('\n'):
#         textobject.textLine(line)
#     c.drawText(textobject)
#     c.save()
# create_dummy_pdf('dummy_policy.pdf', dummy_pdf_content)

# Ensure the dummy PDF exists for the loader to work in a real scenario.
# For this demonstration, we'll just define the path and assume it's there.

pdf_path = 'dummypolicyparents.pdf' # Corrected file name

# Load the documents
# In a real setup, you'd handle potential FileNotFoundError
try:
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from {pdf_path}")
    print(f"Sample content from first page:\n{documents[0].page_content[:500]}...")
except FileNotFoundError:
    print(f"Error: The file '{pdf_path}' was not found. Please ensure it exists or create it.")
    print("Skipping document loading for this step.")
    documents = [] # Empty list to allow pipeline to continue for demonstration

Loaded 2 pages from dummypolicyparents.pdf
Sample content from first page:
Health and Wellbeing 
Dummy Policy 
Statutory Framework for the Early Years Foundation Stage  
Quote Reference: 3:44  
‘The provider must promote the good health of children attending the setting’ 
 
Purpose of the Policy 
The purpose of the policy is to ensure parents/carers are clear on when and how 
dummies will be used in the Family Centre ’s. The policy also gives guidance on the 
impact dummy use can have on a child’s development. 
 
We Aim to: 
 Raise parents/carers awareness of the posi...


2. Document Chunking

Long documents need to be split into smaller, semantically coherent chunks to fit into the LLM's context window and to ensure relevant information can be retrieved effectively. RecursiveCharacterTextSplitter is a good choice for this.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

if documents:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,        # Max characters per chunk
        chunk_overlap=200,      # Overlap between chunks to maintain context
        length_function=len,
        add_start_index=True,
    )
    chunks = text_splitter.split_documents(documents)
    print(f"\nSplit into {len(chunks)} chunks.")
    print(f"Sample chunk:\n{chunks[0].page_content[:500]}...")
else:
    print("No documents to chunk. Please ensure 'dummy_policy.pdf' is present or fix the loading issue.")
    chunks = []


Split into 6 chunks.
Sample chunk:
Health and Wellbeing 
Dummy Policy 
Statutory Framework for the Early Years Foundation Stage  
Quote Reference: 3:44  
‘The provider must promote the good health of children attending the setting’ 
 
Purpose of the Policy 
The purpose of the policy is to ensure parents/carers are clear on when and how 
dummies will be used in the Family Centre ’s. The policy also gives guidance on the 
impact dummy use can have on a child’s development. 
 
We Aim to: 
 Raise parents/carers awareness of the posi...


3. Embedding and Vector Store Creation

Each chunk is converted into a numerical vector (embedding). These embeddings are then stored in a vector database (FAISS in this case) for efficient similarity search. We'll use a HuggingFaceEmbeddings model for this.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

vectorstore = None
if chunks:
    # Initialize the embedding model
    # You can choose different models from Hugging Face or other providers
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Create a FAISS vector store from the document chunks and their embeddings
    vectorstore = FAISS.from_documents(chunks, embeddings)
    print("\nVector store created successfully.")
else:
    print("Cannot create vector store without chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Vector store created successfully.


### 4. Retrieval and Generation (RAG Chain)

Finally, we'll put it all together. When a user asks a question:
1.  The question is embedded.
2.  Relevant chunks are retrieved from the vector store.
3.  These chunks, along with the question, are passed to an LLM to generate a grounded answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
# from langchain_openai import ChatOpenAI # Uncomment and use if using OpenAI
# from langchain_community.llms import Ollama # Uncomment and use if using Ollama
# from langchain_google_genai import ChatGoogleGenerativeAI # Uncomment and use if using Google Gemini

# --- Placeholder for your LLM ----
# Replace with your actual LLM instantiation.
# For example, if using OpenAI:
# llm = ChatOpenAI(model="gpt-3.5-turbo", api_key="YOUR_OPENAI_API_KEY")
# If using Google Gemini:
# llm = ChatGoogleGenerativeAI(model="gemini-pro", google_api_key="YOUR_GEMINI_API_KEY")
# If using Ollama (local LLM):
# llm = Ollama(model="llama2")

# For this demonstration, we'll use a dummy LLM function if no actual LLM is configured.
class DummyLLM:
    def invoke(self, prompt_value): # Renamed parameter for clarity
        # Convert the PromptValue object to a string before processing
        prompt_text = prompt_value.to_string()
        if "waiting period" in prompt_text.lower():
            return "A 30-day waiting period applies for all non-accidental medical claims and a 90-day waiting period for pre-existing conditions." \
                   "This information is retrieved from the 'Waiting Periods' section of the policy."
        elif "dental" in prompt_text.lower():
            return "Dental treatment is covered for accidental injury only, up to $1,000, as stated in Section 1.2 of the policy."
        elif "documents needed" in prompt_text.lower():
            return "Required documents for claim submission include: claim form, medical reports, original bills, and proof of payment, as per Section 4.2."
        else:
            return f"I'm a dummy LLM. I received: '{prompt_text}'. I can provide general answers based on the retrieved context if specific keywords are found."

llm_instance = DummyLLM()
llm = RunnableLambda(llm_instance.invoke)
print("\nDummy LLM initialized for demonstration. Please replace with a real LLM for full functionality.")

# Define the RAG prompt template
rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant for an insurance company. Use the following retrieved policy excerpts to answer the question. If the information is not present, state that you cannot answer based on the provided context. Always quote the source section or page if available."),
    ("human", "Context: {context}\nQuestion: {question}")
])

if vectorstore:
    retriever = vectorstore.as_retriever()

    # Construct the RAG chain
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}  # Retrieve context based on question
        | rag_prompt_template                                       # Format into prompt
        | llm                                                       # Pass to LLM
        | StrOutputParser()                                         # Parse LLM output
    )

    # Example questions
    questions = [
        "What is the waiting period for new policies?",
        "Is dental treatment covered?",
        "What documents are needed for a car accident claim?", # Intentional out-of-context question
        "What is the capital of France?",
        "What is total claim limit?"
    ]

    print("\n--- Running RAG pipeline with example questions ---")
    for q in questions:
        print(f"\nQuestion: {q}")
        response = rag_chain.invoke(q)
        print(f"Answer: {response}")
        print("---------------------------------------------------")
else:
    print("Cannot run RAG chain because vector store was not created due to previous errors.")


Dummy LLM initialized for demonstration. Please replace with a real LLM for full functionality.

--- Running RAG pipeline with example questions ---

Question: What is the waiting period for new policies?
Answer: A 30-day waiting period applies for all non-accidental medical claims and a 90-day waiting period for pre-existing conditions.This information is retrieved from the 'Waiting Periods' section of the policy.
---------------------------------------------------

Question: Is dental treatment covered?
Answer: Dental treatment is covered for accidental injury only, up to $1,000, as stated in Section 1.2 of the policy.
---------------------------------------------------

Question: What documents are needed for a car accident claim?
Answer: I'm a dummy LLM. I received: 'System: You are an AI assistant for an insurance company. Use the following retrieved policy excerpts to answer the question. If the information is not present, state that you cannot answer based on the provided conte